In [1]:
import os
import re
import pandas as pd
from pathlib import Path

#### Practice problem 1.10.4

Download the CMU Seminar Announcements dataset and uncompress it. The dataset consists of multiple files, whereas each file represents a seminar announcement. Write a program that reads every file and tries to extract the speaker, title, start-time, end-time and the location of the seminar. Help yourself with regular expressions and libraries mentioned above. Store all the extracted data into a Pandas data frame and lastly, export all the data into a single CSV file. In addition, compute your success rate in extraction of specific fields based on the manual tags from the documents.

In [2]:
# Example of a file's content
with open('data/sa-tagged/cmu.andrew.academic.sds.seminars-56_0', 'r', encoding='utf-8') as file:
    content = file.read()
    print(content)

<0.2.5.95.11.00.22.cd01+@andrew.cmu.edu.0>
Type:     cmu.andrew.academic.sds.seminars
Topic:    SKVORETZ Seminar
Dates:    4-May-95
Time:     <stime>4:00</stime> - <etime>5:30</etime>
PostedBy: Carole Deaunovich on 2-May-95 at 11:00 from andrew.cmu.edu
Abstract: 

<paragraph><sentence><speaker>Professor John Skvoretz</speaker>, U. of South Carolina, Columbia, will present
a seminar entitled "Embedded Commitment," on Thursday, May 4th from
<stime>4</stime>-<etime>5:30</etime> in <location>PH 223D</sentence></location>.</paragraph>


In [3]:
directory = 'data/sa-tagged'

In [4]:
files = os.listdir(directory)
print("Files in directory:", files)

Files in directory: ['#cmu.andrew.org.cfa.cfa-today-516_1#', 'cmu.andrew.academic.bio-242_0', 'cmu.andrew.academic.met-e-19_1', 'cmu.andrew.academic.sds.seminars-56_0', 'cmu.andrew.assocs.gso-1415_1', 'cmu.andrew.assocs.UEA-142_0', 'cmu.andrew.official.career-center-1495_0', 'cmu.andrew.official.cmu-news-2450_0', 'cmu.andrew.official.cmu-news-2457_0', 'cmu.andrew.official.cmu-news-2477_0', 'cmu.andrew.official.cmu-news-2513_0', 'cmu.andrew.official.cmu-news-2516_0', 'cmu.andrew.official.cmu-news-2527_0', 'cmu.andrew.official.cmu-news-2611_0', 'cmu.andrew.official.cmu-news-2627_0', 'cmu.andrew.official.cmu-news-2633_0', 'cmu.andrew.official.cmu-news-2674_0', 'cmu.andrew.official.cmu-news-2680_0', 'cmu.andrew.official.cmu-news-2737_0', 'cmu.andrew.official.cmu-news-2752_0', 'cmu.andrew.official.cmu-news-2811_0', 'cmu.andrew.official.cmu-news-2912_1', 'cmu.andrew.org.ACE-NIP-9_1', 'cmu.andrew.org.cfa.cfa-today-19_0', 'cmu.andrew.org.cfa.cfa-today-212_0', 'cmu.andrew.org.cfa.cfa-today-516_

In [8]:
# List all files in the directory   
path = Path(directory)
files = list(path.iterdir())  # list of Path objects

for file_path in files:
    if file_path.is_file():
        with open(file_path, 'r') as file:
            content = file.read()
            print(f"Content of {file_path.name}:")
            print(content[:100])  # preview

Content of #cmu.andrew.org.cfa.cfa-today-516_1#:
<0.15.9.94.22.29.02.sm6t+@andrew.cmu.edu.1>
Type:     cmu.andrew.org.cfa.cfa-today
Topic:    1994 Ar
Content of cmu.andrew.academic.bio-242_0:
<0.26.4.95.11.09.31.hf08+@andrew.cmu.edu.0>
Type:     cmu.andrew.academic.bio
Topic:    "MHC Class I
Content of cmu.andrew.academic.met-e-19_1:
<0.17.3.95.09.17.09.ja3q+@andrew.cmu.edu.1>
Type:     cmu.andrew.academic.met-e
Topic:    Career Tal
Content of cmu.andrew.academic.sds.seminars-56_0:
<0.2.5.95.11.00.22.cd01+@andrew.cmu.edu.0>
Type:     cmu.andrew.academic.sds.seminars
Topic:    SKVO
Content of cmu.andrew.assocs.gso-1415_1:
<0.28.4.95.10.13.54.rf51+@andrew.cmu.edu.1>
Type:     cmu.andrew.assocs.gso
Topic:    UTC Summer Sem
Content of cmu.andrew.assocs.UEA-142_0:
<0.15.4.95.15.11.55.rudibear+@CMU.EDU.0>
Type:     cmu.andrew.assocs.UEA
Topic:    Re: entreprenuers
Content of cmu.andrew.official.career-center-1495_0:
<0.23.1.95.12.06.23.kl0y+@andrew.cmu.edu.0>
Type:     cmu.andrew.official.ca

In [18]:
# regex for the required fields
speaker_pattern = r'<speaker>(.*?)</speaker>'
title_pattern = r'entitled\s*"([^"]*?)"'
start_time_pattern = r'<stime>(.*?)</stime>'
end_time_pattern = r'<etime>(.*?)</etime>'
location_pattern = r'<location>(.*?)</location>'

In [19]:
data = []
for file_path in files:
    if file_path.is_file():
        with open(file_path, 'r') as file:
            content = file.read()
            
            speaker = re.search(speaker_pattern, content)
            title = re.search(title_pattern, content)
            start_time = re.search(start_time_pattern, content)
            end_time = re.search(end_time_pattern, content)
            location = re.search(location_pattern, content)

            # Get the matched strings or set them to None if not found
            speaker = speaker.group(1) if speaker else None
            title = title.group(1) if title else None
            start_time = start_time.group(1) if start_time else None
            end_time = end_time.group(1) if end_time else None
            location = location.group(1).replace('</sentence>', '').strip() if location else None
            
            data.append([speaker, title, start_time, end_time, location])
            
data[:5]  # preview

[['MERCE CUNNINGHAM', None, '5:00 PM', None, 'DH 2315'],
 ['Dr. Jeffrey D. Hermes',
  None,
  '3:30 PM',
  None,
  'Mellon Institute Conference Room'],
 [None, None, '12:00 PM', None, None],
 ['Professor John Skvoretz',
  'Embedded Commitment,',
  '4:00',
  '5:30',
  'PH 223D'],
 [None, None, '2:00', '4:00 PM', 'Carnegie Conference Room, Warner Hall']]

In [21]:
df = pd.DataFrame(data, columns=['Speaker', 'Title', 'Start Time', 'End Time', 'Location'])

df.to_csv('data/seminars_data.csv', index=False)